# ENEC 2026 · Day 2 · Colab Setup and Your First Prompts in Code

**Time:** about 40 minutes  |  **Runs in:** Google Colab (free tier is fine)  |  **Needs:** OpenAI key in Colab Secrets (`OPENAI_API_KEY`)

**Data:** a small slice of the synthetic *Nuclear Enterprise 360* dataset, focused on **Asset A-001** (a cooling-water pump). All data is synthetic training data.

**Goal:** Set up Colab and OpenAI once. Then re-run, in Python, the structured prompts you practised in Copilot on Day 1: grounding, structure, examples and boundaries.

> Training notice: nothing in this notebook is an engineering diagnosis or a maintenance authorisation. The AI supports a qualified human reviewer.

## Where this fits

**Day 1** you used Microsoft Copilot and a structured prompt (role, task, context, evidence, constraints, format, check) on the asset workbooks. **Today** the same ideas run in code, so an application can use them:

| Day 1 in Copilot | Day 2 in Python |
|---|---|
| Role and constraints | the **system** message |
| Evidence (a table or a document) | text you put in the prompt, later **retrieved** by RAG |
| Format | a **structured response** (JSON) your code can read |
| Check | tests your code can run |

This first notebook takes about 40 minutes: set up once, then six short experiments on one asset record.

## Step 1. Put your OpenAI key in Colab Secrets (do this once)

1. Click the **key icon** in the left sidebar of Colab.
2. Click **Add new secret**. Name: `OPENAI_API_KEY`. Value: your key.
3. Switch **Notebook access** ON for this notebook.
4. Never paste the key into a code cell. Never share a notebook with the key typed into it.

In [ ]:
# Install what Colab does not already have (about 20 seconds)
%pip install -q -U openai 

In [ ]:
# Get the course files (data, documents, skills) and the helper module.
# In Colab this clones the public course repo into /content/enecoct2026.
import os, sys, subprocess

REPO_URL = "https://github.com/Decoding-Data-Science/enecoct2026"
IN_COLAB = os.path.isdir("/content") and "google.colab" in sys.modules
ROOT = "/content/enecoct2026" if IN_COLAB else os.getcwd()

if IN_COLAB and not os.path.isdir(ROOT):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, ROOT], capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(
            "Could not clone the course repo:\n" + r.stderr[-300:] +
            "\nFallback: upload enecoct2026.zip in the Files panel (left sidebar), "
            "run  !unzip -q enecoct2026.zip -d /content  and run this cell again."
        )

sys.path.insert(0, os.path.join(ROOT, "colab"))
import enec_colab as enec
print("Course files ready at:", enec.ROOT)

In [ ]:
# Connect to OpenAI. The key comes from Colab Secrets and is never printed.
client = enec.get_client()
MODEL = enec.pick_model(client)     # first available of: gpt-5.4-mini, gpt-5-mini, gpt-4o-mini

In [ ]:
# First call. If this prints a sentence, your setup works.
print(enec.ask(client, "In one sentence, say hello to the ENEC training group."))

## The same prompting ideas, now in code

A prompt is an interface contract between your code and the model. We will test five ideas on the **same asset record** (compare with the Day 1 guide, `Day_1_Copilot_and_Data_Understanding/PROMPT_ENGINEERING_GUIDE.md`):

1. No context vs grounded context
2. Prompt anatomy (role, task, context, format, constraints)
3. Structured output (JSON your code can use)
4. Few-shot examples
5. Boundaries (what the model must not do)

In [ ]:
import pandas as pd, json
tables = enec.load_tables()
a360 = tables["asset_360"]
row = a360[a360.asset_id == "A-001"].iloc[0].to_dict()
row

## Experiment 1. Ask without context

The model has never seen this synthetic asset. Watch what it does.

In [ ]:
q = "What is the current health score and recommended action for asset A-001?"
print(enec.ask(client, q))

**Discuss:** Did it admit it does not know, or did it invent something plausible? An invented but confident answer is a *hallucination*.

## Experiment 2. Same question, grounded in the record

In [ ]:
grounded = f"""Use ONLY this record to answer.

RECORD (from table asset_360):
{json.dumps(row, default=str, indent=2)}

QUESTION: {q}"""
print(enec.ask(client, grounded))

## Experiment 3. Prompt anatomy

A reliable prompt has five parts. Name them in your prompt and the output becomes predictable.

In [ ]:
ROLE        = "You are a reliability engineering analyst supporting a qualified human reviewer."
TASK        = "Summarise the current condition of asset A-001 for a morning stand-up."
CONTEXT     = json.dumps(row, default=str)
FORMAT      = "Exactly three bullet points: Status, Why it matters, Suggested human next step. Max 25 words each."
CONSTRAINTS = "Use only the CONTEXT. If something is not in the CONTEXT, write 'not in the data'. Do not authorise any work."

prompt = f"""ROLE: {ROLE}
TASK: {TASK}
CONTEXT: {CONTEXT}
FORMAT: {FORMAT}
CONSTRAINTS: {CONSTRAINTS}"""
print(enec.ask(client, prompt))

**Try it:** change `FORMAT` to a table, then to one sentence. Change `CONSTRAINTS` and see what breaks.

## Experiment 4. Structured output (JSON your code can use)

Free text is for people. Code needs structure. Ask for JSON and parse it.

In [ ]:
wo = tables["work_orders"]
open_wo = wo[(wo.asset_id == "A-001") & (wo.status.isin(["OPEN", "IN_PROGRESS", "DEFERRED"]))]
wo_text = open_wo[["work_order_id", "work_type", "priority", "created_date", "problem_description"]].to_csv(index=False)

system = "You convert work-order tables into JSON. Return a JSON object: {\"items\": [{\"work_order_id\": str, \"priority\": str, \"issue_type\": one of [VIBRATION, TEMPERATURE, NOISE, LEAKAGE, CALIBRATION, PREVENTIVE, OTHER], \"needs_urgent_attention\": bool}]}"
raw = enec.ask(client, wo_text, system=system, json_mode=True)
items = json.loads(raw).get("items", [])
pd.DataFrame(items)

## Experiment 5. Few-shot examples

Show the model 3 labelled examples, then let it label the rest. We label every distinct problem description in the full work-order table (1,500 rows).

In [ ]:
descs = sorted(wo.problem_description.dropna().unique().tolist())
print(len(descs), "distinct descriptions")

LABELS = ["VIBRATION", "TEMPERATURE", "NOISE", "LEAKAGE", "CALIBRATION", "PREVENTIVE", "OTHER"]
examples = """Examples:
"Abnormal vibration observed during routine monitoring." -> VIBRATION
"Temperature trend exceeded expected operating band." -> TEMPERATURE
"Preventive maintenance interval reached." -> PREVENTIVE"""

system = (f"Label each description with one of {LABELS}. {examples}\n"
          "Return JSON: {\"labels\": {\"0\": LABEL, \"1\": LABEL, ...}} using the numbers given.")
numbered = "\n".join(f"{i}: {d}" for i, d in enumerate(descs))
out = json.loads(enec.ask(client, numbered, system=system, json_mode=True)).get("labels", {})
labelled = pd.DataFrame({"description": descs, "label": [out.get(str(i), "UNKNOWN") for i in range(len(descs))]})
labelled

In [ ]:
# Use the labels: how many work orders per label, and how many are still open?
lab_map = dict(zip(labelled.description, labelled.label))
wo2 = wo.assign(label=wo.problem_description.map(lab_map))
wo2.groupby("label").agg(work_orders=("work_order_id", "count"),
                         open_items=("status", lambda s: s.isin(["OPEN", "IN_PROGRESS", "DEFERRED"]).sum()))

## Experiment 6. Boundaries

The most important prompt is the one that says what the model must **not** do.

In [ ]:
BOUNDARY = ("You support a qualified human reviewer. You never authorise maintenance, shutdown or replacement. "
            "When asked to decide, explain what evidence exists, what is missing, and who should decide.")
risky = f"""RECORD: {json.dumps(row, default=str)}
A manager says: 'Just tell us. Should we shut down A-001 right now and replace the pump?'"""

print("--- without boundary ---")
print(enec.ask(client, risky))
print("\n--- with boundary ---")
print(enec.ask(client, risky, system=BOUNDARY))

## Takeaways

- **Ground it:** put the facts in the prompt, or the model will fill gaps with guesses.
- **Name the parts:** role, task, context, format, constraints.
- **Ask for JSON** when code consumes the answer, and handle missing keys.
- **Show examples** when labels must be consistent.
- **Set boundaries** and keep a human in charge of consequential decisions.

## Exercise

Write a prompt that produces a 2-line shift-handover note for the **three lowest-health assets** in `asset_360`. Use `a360.sort_values("health_score").head(3)` as context, force JSON output, and add one constraint you think the night shift would want.